In [ ]:
from pathlib import Path

input_root = Path("/kaggle/input")

data_candidates = [
    p.parent for p in input_root.rglob("features.csv")
    if (p.parent / "series.npy").is_file()
    and (p.parent / "scores" / "isolation_forest.csv").is_file()
]

project_candidates = [
    p.parent for p in input_root.rglob("pyproject.toml")
    if (p.parent / "src" / "cosmic_black_swan" / "transformer.py").is_file()
]

print("Data locations:", data_candidates)
print("Project locations:", project_candidates)

if not data_candidates or not project_candidates:
    raise RuntimeError(
        "Input missing. Attach the successful notebook Version as an input, "
        "and attach the Cosmic Black Swan project source if needed."
    )

DATA_INPUT = data_candidates[0]
PROJECT_INPUT = project_candidates[0]
print("Using data:", DATA_INPUT)
print("Using project:", PROJECT_INPUT)

In [ ]:
import shutil
import subprocess
import sys

DATA_ROOT = Path("/kaggle/working/cbs-transformer-data")
PROJECT_ROOT = Path("/kaggle/working/cbs-project-src")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Copy only what Transformer training needs; this avoids copying the raw FITS files.
for name in ["features.csv", "series.npy"]:
    shutil.copy2(DATA_INPUT / name, DATA_ROOT / name)

# Keep baseline results available for comparison after training.
(DATA_ROOT / "scores").mkdir(exist_ok=True)
for name in ["isolation_forest.csv", "conformal_test.csv",
             "blindspot_sensitivity_map.csv"]:
    source = DATA_INPUT / "scores" / name
    if source.is_file():
        shutil.copy2(source, DATA_ROOT / "scores" / name)

shutil.copytree(PROJECT_INPUT, PROJECT_ROOT, dirs_exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT) + "[deep]"],
    check=True,
)

import torch
print("Features:", (DATA_ROOT / "features.csv").exists())
print("Series:", (DATA_ROOT / "series.npy").exists())
print("GPU available:", torch.cuda.is_available())

In [ ]:
import subprocess

subprocess.run([
    "cbs", "train-transformer",
    "--data-root", str(DATA_ROOT),
    "--epochs", "20",
    "--batch-size", "64",
], check=True)

score_file = DATA_ROOT / "scores" / "transformer_holdout.csv"
model_file = DATA_ROOT / "models" / "masked_patch_transformer.pt"

print("Transformer scores saved:", score_file.exists(), score_file)
print("Trained model saved:", model_file.exists(), model_file)